# Workload & Application Security Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-39 — Container Least Privilege**: Least privilege is enforced for containers and pods. Evidence on-cluster: how many users / service accounts are bound to the `privileged` and `anyuid` SecurityContextConstraints, and whether the `restricted-v2` SCC is present (the OCP 4.18 default-restricted profile that drops capabilities, blocks privilege escalation, and enforces a non-root run-as strategy).
- **OCP-40 — Security Context Constraint (SCC) Enforcement**: SCC standards are configured and enforced. Evidence on-cluster: the `privileged` SCC is bound only to platform / `system:` service accounts and groups (no end-user groups), `restricted-v2` exists with `allowPrivilegedContainer=false`, `allowPrivilegeEscalation=false`, `requiredDropCapabilities=ALL`, and a non-`RunAsAny` run-as strategy.
- **OCP-41 — Workload Resource Quotas**: Resource limitations are enforced for scheduling workloads. Evidence on-cluster: presence of `ResourceQuota` (or `ClusterResourceQuota`) and `LimitRange` records that constrain CPU / memory / pod counts on workload namespaces.
- **OCP-42 — Trusted Image Enforcement**: Cluster configuration prevents the use of unapproved images. Evidence on-cluster: `image.config.openshift.io/cluster` restricts `registrySources.allowedRegistries`, at least one `ClusterImagePolicy` requires signature verification, an `ImageContentSourcePolicy` / `ImageDigestMirrorSet` pins mirrors, and an image-policy admission webhook is present.
- **OCP-43 — Pod Security Context**: Security context is leveraged for pods, containers, and storage. Evidence on-cluster: every user namespace carries Pod Security Admission labels (`pod-security.kubernetes.io/enforce`) at `baseline` or `restricted` (no `privileged`, no missing labels).
- **OCP-44 — Runtime Security**: Container runtime security best practices are enforced. Evidence on-cluster: a runtime threat-detection product is installed and running — Red Hat Advanced Cluster Security (RHACS / StackRox) Secured Cluster, Falco, NeuVector, Sysdig, or equivalent.
- **OCP-45 — Logical Project Isolation**: Each project / namespace is logically isolated. Evidence on-cluster: every user namespace has a default-deny `NetworkPolicy`, a `ResourceQuota` and `LimitRange`, an ownership label, and at least one project-scoped `RoleBinding`.
- **OCP-46 — Encryption at Rest**: Data at rest on the cluster is encrypted. Evidence on-cluster: `apiserver.config.openshift.io/cluster` enables etcd encryption (`aescbc` or `aesgcm`), default `StorageClass` parameters request encryption (and ideally a KMS key), and worker-node `MachineConfig` resources configure LUKS / Tang / Clevis for root-volume encryption.
- **OCP-47 — Build / Source-to-Image (S2I) Policy**: Build pipelines on the cluster follow a hardened policy. Evidence on-cluster: a cluster-scoped `build.config.openshift.io/cluster` supplies build defaults (image labels, resources), no `BuildConfig` uses the `Custom` strategy with the docker socket exposed, and every `ImageStream` has `spec.lookupPolicy.local=true`.
- **OCP-48 — Image Signing & Verification**: Container images are signed and signatures are verified before deployment. Evidence on-cluster: at least one `ClusterImagePolicy` requires a signature (`PublicKey` or `FulcioCAWithRekor`) and scopes registries, build pipelines invoke `cosign` / `notation`, and the image registry is configured with a sigstore signature store.
- **OCP-49 — Ephemeral Storage Limits**: Ephemeral storage is bounded so pods cannot exhaust node disks. Evidence on-cluster: every user namespace has a `LimitRange` default request / default limit for `ephemeral-storage`, a `ResourceQuota` caps `requests.ephemeral-storage`, and no pod uses an `emptyDir` volume without `sizeLimit`.
- **OCP-50 — Admission Controller Hardening**: Built-in and webhook admission controllers are configured to fail safe. Evidence on-cluster: the OCP 4.18 default-on hardening plugins (`LimitRanger`, `ResourceQuota`, `PodSecurity`, `NodeRestriction`, `MutatingAdmissionWebhook`, `ValidatingAdmissionWebhook`) are not explicitly disabled, and no webhook combines `failurePolicy=Ignore` with `sideEffects=Unknown`.


In [ ]:
import os
import sys

import pandas as pd

# Ensure the repo root is importable before loading project modules.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from notebook_style import bootstrap, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

# bootstrap() adds ../datastore to sys.path, which is required before
# importing schema.models below.
session, engine = bootstrap()

from schema.models import (  # noqa: E402
    AdmissionControllerHardening,
    BuildS2iPolicy,
    Cluster,
    EncryptionAtRest,
    EphemeralStorageLimits,
    ImageSigningVerification,
    LogicalProjectIsolation,
    PodSecurityAdmission,
    SccPrivileged,
    TrustedImageEnforcement,
    VulnerabilityRuntimeDetection,
    WorkloadResourceQuota,
)

print("Connected to:", engine.url)

## Cluster Inventory


In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

In [ ]:
def _split_semi(value):
    """Split a ';'-delimited cell into a list of non-empty trimmed strings."""
    if value is None:
        return []
    return [v.strip() for v in str(value).split(";") if v.strip()]


def _is_system_principal(principal: str) -> bool:
    """True if the user/group is a platform-managed identity rather than an end user / team."""
    if not principal:
        return False
    return principal.startswith("system:")

---
## OCP-39: Container Least Privilege

*Least privilege is enforced for containers and pods.*

On-cluster evidence is the **bind count on the `privileged` SCC** (which lets pods run as root, share host namespaces, and get any capability), the **bind count on the `anyuid` SCC** (which lifts only the run-as-user requirement but still permits running as UID 0), and the **presence of `restricted-v2`** — the OCP 4.18 default SCC that drops all capabilities, blocks privilege escalation, and forces a non-root run-as range. A cluster with many `privileged` bindings, broad `anyuid` use, or a missing `restricted-v2` cannot guarantee least privilege for workloads.

### Privileged-class SCC records (per cluster)


In [ ]:
df_ocp39 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        SccPrivileged.name,
        SccPrivileged.allow_privileged_container,
        SccPrivileged.allow_privilege_escalation,
        SccPrivileged.users_count,
        SccPrivileged.groups_count,
        SccPrivileged.users,
        SccPrivileged.groups,
    )
    .join(Cluster, SccPrivileged.cluster_id == Cluster.id)
    .filter(SccPrivileged.name.in_(["privileged", "anyuid", "restricted-v2"]))
    .order_by(Cluster.cluster_name, SccPrivileged.name)
    .statement,
    engine,
)
style_table(df_ocp39, caption="OCP-39: Privileged-class SCC bindings per cluster")

### OCP-39: Compliance flags

Per-cluster flags derived from the raw records. Thresholds reflect a baseline
OCP 4.18 install where `system:admin`, `system:cluster-admins`, and a small
fixed set of platform service accounts (build controller, machine-api
controllers, monitoring) are expected on `privileged` / `anyuid`.

A cluster is considered **compliant** when:

- `restricted_v2_present` — the cluster has a `restricted-v2` SCC.
- `privileged_users_within_baseline` — `privileged` SCC has `users_count <= 3` (only platform service accounts).
- `anyuid_users_within_baseline` — `anyuid` SCC has `users_count <= 1`.

Only clusters that fail one or more of these checks are displayed.


In [ ]:
PRIV_USERS_BASELINE = 3
ANYUID_USERS_BASELINE = 1

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp39[df_ocp39["cluster_name"] == cluster_name]
    by_name = {r["name"]: r for _, r in sub.iterrows()}

    priv = by_name.get("privileged")
    anyuid = by_name.get("anyuid")
    restricted = by_name.get("restricted-v2")

    priv_users = (
        int(priv["users_count"])
        if priv is not None and pd.notna(priv["users_count"])
        else 0
    )
    anyuid_users = (
        int(anyuid["users_count"])
        if anyuid is not None and pd.notna(anyuid["users_count"])
        else 0
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "restricted_v2_present": restricted is not None,
            "privileged_users_count": priv_users,
            "privileged_users_within_baseline": priv_users <= PRIV_USERS_BASELINE,
            "anyuid_users_count": anyuid_users,
            "anyuid_users_within_baseline": anyuid_users <= ANYUID_USERS_BASELINE,
        }
    )

df_ocp39_flags = pd.DataFrame(rows)
df_ocp39_flags["compliant"] = (
    df_ocp39_flags["restricted_v2_present"]
    & df_ocp39_flags["privileged_users_within_baseline"]
    & df_ocp39_flags["anyuid_users_within_baseline"]
)
df_ocp39_noncompliant = df_ocp39_flags[~df_ocp39_flags["compliant"]].copy()
print(
    f"{len(df_ocp39_noncompliant)} of {len(df_ocp39_flags)} cluster(s) non-compliant for OCP-39"
)
style_table(df_ocp39_noncompliant, caption="OCP-39: Non-compliant clusters")

---
## OCP-40: Security Context Constraint (SCC) Enforcement

*Security Context Constraint (SCC) standards are configured and enforced.*

On-cluster evidence is the **policy posture of the `restricted-v2` SCC** (must reject privileged containers, privilege escalation, host namespaces, and require dropping all capabilities), and the **scope of the `privileged` SCC's `groups` binding** — only `system:`-prefixed groups (e.g. `system:cluster-admins`) are permitted; binding `privileged` to a non-`system:` group means end users / team groups can launch privileged pods.

### Privileged & restricted-v2 SCC posture (per cluster)


In [ ]:
df_ocp40 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        SccPrivileged.name,
        SccPrivileged.allow_privileged_container,
        SccPrivileged.allow_privilege_escalation,
        SccPrivileged.allow_host_network,
        SccPrivileged.allow_host_pid,
        SccPrivileged.allow_host_ipc,
        SccPrivileged.required_drop_capabilities,
        SccPrivileged.run_as_user_type,
        SccPrivileged.groups,
    )
    .join(Cluster, SccPrivileged.cluster_id == Cluster.id)
    .filter(SccPrivileged.name.in_(["privileged", "restricted-v2"]))
    .order_by(Cluster.cluster_name, SccPrivileged.name)
    .statement,
    engine,
)
style_table(df_ocp40, caption="OCP-40: SCC enforcement posture (per cluster)")

### OCP-40: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `restricted_v2_present` — the cluster has a `restricted-v2` SCC.
- `restricted_v2_no_priv_container` — `restricted-v2.allow_privileged_container` is `false`.
- `restricted_v2_no_priv_escalation` — `restricted-v2.allow_privilege_escalation` is `false`.
- `restricted_v2_drops_all_caps` — `restricted-v2.required_drop_capabilities` is `ALL`.
- `restricted_v2_run_as_enforced` — `restricted-v2.run_as_user_type` is not `RunAsAny` (i.e. one of `MustRunAsRange`, `MustRunAsNonRoot`, `MustRunAs`).
- `privileged_groups_system_only` — every group bound to the `privileged` SCC is `system:`-prefixed (no end-user / team groups).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp40[df_ocp40["cluster_name"] == cluster_name]
    by_name = {r["name"]: r for _, r in sub.iterrows()}

    priv = by_name.get("privileged")
    restricted = by_name.get("restricted-v2")

    if restricted is not None:
        r_no_priv = str(restricted["allow_privileged_container"]).lower() == "false"
        r_no_esc = str(restricted["allow_privilege_escalation"]).lower() == "false"
        r_drop_all = (
            str(restricted["required_drop_capabilities"] or "").upper() == "ALL"
        )
        r_run_as = str(restricted["run_as_user_type"] or "")
        r_run_as_ok = r_run_as not in ("", "RunAsAny")
    else:
        r_no_priv = r_no_esc = r_drop_all = r_run_as_ok = False
        r_run_as = ""

    if priv is not None:
        priv_groups = _split_semi(priv["groups"])
        non_system_groups = [g for g in priv_groups if not _is_system_principal(g)]
        priv_groups_ok = len(non_system_groups) == 0
    else:
        priv_groups = []
        non_system_groups = []
        priv_groups_ok = False

    rows.append(
        {
            "cluster_name": cluster_name,
            "restricted_v2_present": restricted is not None,
            "restricted_v2_no_priv_container": r_no_priv,
            "restricted_v2_no_priv_escalation": r_no_esc,
            "restricted_v2_drops_all_caps": r_drop_all,
            "restricted_v2_run_as_type": r_run_as,
            "restricted_v2_run_as_enforced": r_run_as_ok,
            "privileged_groups": ";".join(priv_groups),
            "privileged_non_system_groups": ";".join(non_system_groups),
            "privileged_groups_system_only": priv_groups_ok,
        }
    )

df_ocp40_flags = pd.DataFrame(rows)
df_ocp40_flags["compliant"] = (
    df_ocp40_flags["restricted_v2_present"]
    & df_ocp40_flags["restricted_v2_no_priv_container"]
    & df_ocp40_flags["restricted_v2_no_priv_escalation"]
    & df_ocp40_flags["restricted_v2_drops_all_caps"]
    & df_ocp40_flags["restricted_v2_run_as_enforced"]
    & df_ocp40_flags["privileged_groups_system_only"]
)
df_ocp40_noncompliant = df_ocp40_flags[~df_ocp40_flags["compliant"]].copy()
print(
    f"{len(df_ocp40_noncompliant)} of {len(df_ocp40_flags)} cluster(s) non-compliant for OCP-40"
)
style_table(df_ocp40_noncompliant, caption="OCP-40: Non-compliant clusters")

---
## OCP-41: Workload Resource Quotas

*Resource limitations are enforced for scheduling workloads.*

On-cluster evidence is the **presence of `ResourceQuota` or `ClusterResourceQuota`** records (which cap CPU, memory, pod / object counts on a namespace or across a tenant) and the **presence of `LimitRange`** records (which set per-Container default and max CPU / memory). A cluster missing either category cannot guarantee that a single namespace or pod is unable to exhaust shared cluster capacity.

### Quota & LimitRange records (per cluster)


In [ ]:
df_ocp41 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        WorkloadResourceQuota.record_type,
        WorkloadResourceQuota.namespace,
        WorkloadResourceQuota.name,
        WorkloadResourceQuota.resource_key,
        WorkloadResourceQuota.hard_limit,
        WorkloadResourceQuota.used,
        WorkloadResourceQuota.limit_type,
    )
    .join(Cluster, WorkloadResourceQuota.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        WorkloadResourceQuota.record_type,
        WorkloadResourceQuota.namespace,
    )
    .statement,
    engine,
)
style_table(df_ocp41, caption="OCP-41: Quota & LimitRange records (per cluster)")

### OCP-41: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `has_resourcequota` — at least one `ResourceQuota` record exists on the cluster.
- `has_limitrange` — at least one `LimitRange` record exists on the cluster.

`has_clusterresourcequota` is reported as evidence (multi-tenant quota envelope) but is not required for the compliance verdict, since not every cluster exposes its workloads to a `ClusterResourceQuota`-managed tenant.

Only clusters that fail one or more of the required checks are displayed.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp41[df_ocp41["cluster_name"] == cluster_name]
    rq_count = int((sub["record_type"] == "resourcequota").sum())
    crq_count = int((sub["record_type"] == "clusterresourcequota").sum())
    lr_count = int((sub["record_type"] == "limitrange").sum())
    rq_namespaces = sorted(
        sub.loc[sub["record_type"] == "resourcequota", "namespace"]
        .dropna()
        .unique()
        .tolist()
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "resourcequota_count": rq_count,
            "clusterresourcequota_count": crq_count,
            "limitrange_count": lr_count,
            "namespaces_with_resourcequota": ";".join(rq_namespaces),
            "has_resourcequota": rq_count > 0,
            "has_clusterresourcequota": crq_count > 0,
            "has_limitrange": lr_count > 0,
        }
    )

df_ocp41_flags = pd.DataFrame(rows)
df_ocp41_flags["compliant"] = (
    df_ocp41_flags["has_resourcequota"] & df_ocp41_flags["has_limitrange"]
)
df_ocp41_noncompliant = df_ocp41_flags[~df_ocp41_flags["compliant"]].copy()
print(
    f"{len(df_ocp41_noncompliant)} of {len(df_ocp41_flags)} cluster(s) non-compliant for OCP-41"
)
style_table(df_ocp41_noncompliant, caption="OCP-41: Non-compliant clusters")

---
## OCP-42: Trusted Image Enforcement

*Cluster configuration to prevent use of unapproved images is enforced.*

On OCP 4.18 the relevant evidence is `image.config.openshift.io/cluster` (`registrySources.allowedRegistries` and friends), `ClusterImagePolicy` / `ImagePolicy` resources that require signature verification, `ImageContentSourcePolicy` / `ImageDigestMirrorSet` records that pin mirrors, and an image-policy admission webhook.


In [ ]:
df_ocp42 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        TrustedImageEnforcement.record_type,
        TrustedImageEnforcement.name,
        TrustedImageEnforcement.detail_1,
        TrustedImageEnforcement.detail_2,
        TrustedImageEnforcement.detail_3,
        TrustedImageEnforcement.detail_4,
        TrustedImageEnforcement.detail_5,
        TrustedImageEnforcement.detail_6,
    )
    .join(Cluster, TrustedImageEnforcement.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, TrustedImageEnforcement.record_type)
    .statement,
    engine,
)
style_table(df_ocp42, caption="OCP-42 — Trusted Image Enforcement (raw records)")

### OCP-42: Compliance flags

A cluster is considered **compliant** when **all** of the following hold:

- `image_config.allowed_registries` is non-empty (registry sources are restricted).
- At least one `cluster_image_policy` (or `image_policy`) record has `signature_required=true`.
- At least one `image_content_source_policy` / mirror-set record exists.
- The `ImagePolicyWebhook` admission plugin is detected (`enabled=true`).


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp42[df_ocp42["cluster_name"] == cluster_name]
    cfg = sub[sub["record_type"] == "image_config"]
    allowed_registries = ""
    if not cfg.empty:
        allowed_registries = cfg.iloc[0]["detail_1"] or ""
    allowed_registries_count = len(_split_semi(allowed_registries))

    pols = sub[sub["record_type"].isin(["cluster_image_policy", "image_policy"])]
    signed_policies = pols[pols["detail_2"].fillna("").str.lower() == "true"]
    signed_policy_count = len(signed_policies)

    icsp = sub[sub["record_type"] == "image_content_source_policy"]
    icsp_count = len(icsp)

    plug = sub[sub["record_type"] == "admission_plugin"]
    webhook_enabled = (not plug.empty) and (
        str(plug.iloc[0]["detail_1"] or "").lower() == "true"
    )

    compliant = (
        allowed_registries_count > 0
        and signed_policy_count >= 1
        and icsp_count >= 1
        and webhook_enabled
    )
    rows.append(
        {
            "cluster_name": cluster_name,
            "allowed_registries_count": allowed_registries_count,
            "signed_image_policies": signed_policy_count,
            "mirror_policies": icsp_count,
            "image_policy_webhook_enabled": webhook_enabled,
            "ocp42_compliant": compliant,
        }
    )
df_ocp42_flags = pd.DataFrame(rows)
style_table(df_ocp42_flags, caption="OCP-42 — Compliance flags by cluster")

---
## OCP-43: Pod Security Context

*Security context is leveraged for pods, containers, and storage.*

Evidence is the per-namespace Pod Security Admission labels — `pod-security.kubernetes.io/enforce`, `audit`, `warn` and their `*-version` counterparts. System namespaces (`openshift-*`, `kube-*`, `default`) are excluded from the verdict because OpenShift platform components legitimately run as `privileged` there.


In [ ]:
df_ocp43 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PodSecurityAdmission.namespace,
        PodSecurityAdmission.is_system_namespace,
        PodSecurityAdmission.enforce_level,
        PodSecurityAdmission.enforce_version,
        PodSecurityAdmission.audit_level,
        PodSecurityAdmission.warn_level,
    )
    .join(Cluster, PodSecurityAdmission.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, PodSecurityAdmission.namespace)
    .statement,
    engine,
)
style_table(df_ocp43, caption="OCP-43 — Pod Security Admission labels (raw records)")

### OCP-43: Compliance flags

A cluster is considered **compliant** when, for every **user** namespace (non-`openshift-*`, non-`kube-*`, non-`default`), `enforce_level` is set to either `baseline` or `restricted` — and **no** user namespace has `enforce_level=privileged` or a missing label.


In [ ]:
ACCEPTABLE_PSA = {"baseline", "restricted"}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp43[df_ocp43["cluster_name"] == cluster_name]
    user_ns = sub[sub["is_system_namespace"].fillna("").str.lower() != "true"]
    user_ns_count = len(user_ns)
    enforce = user_ns["enforce_level"].fillna("").str.lower()
    restricted_count = int((enforce == "restricted").sum())
    baseline_count = int((enforce == "baseline").sum())
    privileged_count = int((enforce == "privileged").sum())
    unset_count = int((enforce == "").sum())
    bad = privileged_count + unset_count
    compliant = (
        user_ns_count > 0
        and bad == 0
        and (restricted_count + baseline_count) == user_ns_count
    )
    rows.append(
        {
            "cluster_name": cluster_name,
            "user_namespaces": user_ns_count,
            "enforce_restricted": restricted_count,
            "enforce_baseline": baseline_count,
            "enforce_privileged": privileged_count,
            "enforce_unset": unset_count,
            "ocp43_compliant": compliant,
        }
    )
df_ocp43_flags = pd.DataFrame(rows)
style_table(df_ocp43_flags, caption="OCP-43 — Compliance flags by cluster")

---
## OCP-44: Runtime Security

*Container runtime security best practices are enforced.*

Evidence is sourced from the `vulnerability-runtime-detection` export — the presence and `installed` status of runtime threat-detection products such as RHACS / StackRox Secured Cluster, Falco, NeuVector, Sysdig Secure, etc.


In [ ]:
df_ocp44 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        VulnerabilityRuntimeDetection.record_type,
        VulnerabilityRuntimeDetection.component_name,
        VulnerabilityRuntimeDetection.status,
        VulnerabilityRuntimeDetection.namespace,
    )
    .join(Cluster, VulnerabilityRuntimeDetection.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, VulnerabilityRuntimeDetection.component_name)
    .statement,
    engine,
)
style_table(df_ocp44, caption="OCP-44 — Runtime detection products (raw records)")

### OCP-44: Compliance flags

A cluster is considered **compliant** when at least one runtime threat-detection product is `status=installed`. Recognised runtime products: `rhacs-secured-cluster`, `falco`, `neuvector`, `sysdig-secure`, `crowdstrike`, `prisma-cloud`, `aqua-enforcer`. (`rhacs-central` and `quay-enterprise` are not runtime threat-detection products and do not count toward this control.)


In [ ]:
RUNTIME_TOOLS = {
    "rhacs-secured-cluster",
    "falco",
    "neuvector",
    "sysdig-secure",
    "crowdstrike",
    "prisma-cloud",
    "aqua-enforcer",
}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp44[df_ocp44["cluster_name"] == cluster_name]
    runtime = sub[sub["component_name"].fillna("").str.lower().isin(RUNTIME_TOOLS)]
    installed = runtime[runtime["status"].fillna("").str.lower() == "installed"]
    installed_tools = sorted(set(installed["component_name"].fillna("").str.lower()))
    compliant = len(installed_tools) >= 1
    rows.append(
        {
            "cluster_name": cluster_name,
            "runtime_tools_installed": ";".join(installed_tools) or "(none)",
            "runtime_tool_count": len(installed_tools),
            "ocp44_compliant": compliant,
        }
    )
df_ocp44_flags = pd.DataFrame(rows)
style_table(df_ocp44_flags, caption="OCP-44 — Compliance flags by cluster")

---
## OCP-45: Logical Project Isolation

*Each project / namespace is logically isolated.*

Evidence is sourced from the `logical-project-isolation` export — per-namespace
flags for default-deny `NetworkPolicy`, `ResourceQuota` / `LimitRange`
enforcement, an ownership label, plus counts of `RoleBindings` and distinct
`ServiceAccounts`. System namespaces (`openshift-*`, `kube-*`, `default`) are
excluded from the verdict because OpenShift platform components legitimately
share access there.


In [ ]:
df_ocp45 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        LogicalProjectIsolation.namespace,
        LogicalProjectIsolation.is_system_namespace,
        LogicalProjectIsolation.has_default_deny_netpol,
        LogicalProjectIsolation.netpol_count,
        LogicalProjectIsolation.has_resourcequota,
        LogicalProjectIsolation.has_limitrange,
        LogicalProjectIsolation.has_owner_label,
        LogicalProjectIsolation.rolebinding_count,
        LogicalProjectIsolation.distinct_serviceaccounts,
    )
    .join(Cluster, LogicalProjectIsolation.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, LogicalProjectIsolation.namespace)
    .statement,
    engine,
)
style_table(df_ocp45, caption="OCP-45 — Logical Project Isolation (raw records)")

### OCP-45: Compliance flags

A cluster is considered **compliant** when, for every **user** namespace
(non-`openshift-*`, non-`kube-*`, non-`default`), **all** of the following
hold:

- `has_default_deny_netpol=true`
- `has_resourcequota=true`
- `has_limitrange=true`
- `has_owner_label=true`
- `rolebinding_count >= 1`


In [ ]:
def _b(s):
    return str(s or "").strip().lower() == "true"


def _i(s):
    try:
        return int(str(s).strip())
    except (ValueError, TypeError):
        return 0


rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp45[df_ocp45["cluster_name"] == cluster_name]
    user_ns = sub[sub["is_system_namespace"].fillna("").str.lower() != "true"]
    user_ns_count = len(user_ns)
    deny_ok = sum(_b(v) for v in user_ns["has_default_deny_netpol"])
    rq_ok = sum(_b(v) for v in user_ns["has_resourcequota"])
    lr_ok = sum(_b(v) for v in user_ns["has_limitrange"])
    owner_ok = sum(_b(v) for v in user_ns["has_owner_label"])
    rb_ok = sum(_i(v) >= 1 for v in user_ns["rolebinding_count"])
    fully_isolated = sum(
        _b(d) and _b(rq) and _b(lr) and _b(o) and _i(rb) >= 1
        for d, rq, lr, o, rb in zip(
            user_ns["has_default_deny_netpol"],
            user_ns["has_resourcequota"],
            user_ns["has_limitrange"],
            user_ns["has_owner_label"],
            user_ns["rolebinding_count"],
        )
    )
    compliant = user_ns_count > 0 and fully_isolated == user_ns_count
    rows.append(
        {
            "cluster_name": cluster_name,
            "user_namespaces": user_ns_count,
            "with_default_deny": deny_ok,
            "with_resourcequota": rq_ok,
            "with_limitrange": lr_ok,
            "with_owner_label": owner_ok,
            "with_rolebinding": rb_ok,
            "fully_isolated": fully_isolated,
            "ocp45_compliant": compliant,
        }
    )
df_ocp45_flags = pd.DataFrame(rows)
style_table(df_ocp45_flags, caption="OCP-45 — Compliance flags by cluster")

---
## OCP-46: Encryption at Rest

*Data at rest on the cluster is encrypted.*

Evidence is sourced from the `encryption-at-rest` export. `record_type` is one
of:

- `etcd_encryption` — `apiserver.config.openshift.io/cluster` `spec.encryption.type`
- `storage_class` — provisioner, `encrypted` parameter, KMS key, default flag
- `machine_config_luks` — LUKS / Tang / Clevis configuration on worker nodes
- `persistent_volume` — sample PVs and the encryption status of their backing `StorageClass`


In [ ]:
df_ocp46 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        EncryptionAtRest.record_type,
        EncryptionAtRest.name,
        EncryptionAtRest.namespace,
        EncryptionAtRest.detail_1,
        EncryptionAtRest.detail_2,
        EncryptionAtRest.detail_3,
        EncryptionAtRest.detail_4,
        EncryptionAtRest.detail_5,
        EncryptionAtRest.detail_6,
    )
    .join(Cluster, EncryptionAtRest.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, EncryptionAtRest.record_type, EncryptionAtRest.name)
    .statement,
    engine,
)
style_table(df_ocp46, caption="OCP-46 — Encryption at Rest (raw records)")

### OCP-46: Compliance flags

A cluster is considered **compliant** when **all** of the following hold:

- The `etcd_encryption` row has `detail_1 in {aescbc, aesgcm}` (a real cipher,
  not the default `identity` no-op).
- The default `storage_class` (`detail_4=true`) has `detail_2=true` (its
  `encrypted` parameter is enabled).
- At least one `machine_config_luks` row is present (LUKS / Tang or LUKS / TPM
  is configured for worker root volumes).


In [ ]:
ETCD_OK = {"aescbc", "aesgcm"}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp46[df_ocp46["cluster_name"] == cluster_name]

    etcd = sub[sub["record_type"] == "etcd_encryption"]
    etcd_type = (etcd.iloc[0]["detail_1"] if not etcd.empty else "") or ""
    etcd_ok = etcd_type.lower() in ETCD_OK

    sc = sub[sub["record_type"] == "storage_class"]
    default_sc = sc[sc["detail_4"].fillna("").str.lower() == "true"]
    default_sc_encrypted = (
        not default_sc.empty
        and str(default_sc.iloc[0]["detail_2"] or "").lower() == "true"
    )
    encrypted_sc_count = int((sc["detail_2"].fillna("").str.lower() == "true").sum())

    luks = sub[sub["record_type"] == "machine_config_luks"]
    luks_count = len(luks)

    pv = sub[sub["record_type"] == "persistent_volume"]
    pv_total = len(pv)
    pv_encrypted = int((pv["detail_2"].fillna("").str.lower() == "true").sum())

    compliant = etcd_ok and default_sc_encrypted and luks_count >= 1
    rows.append(
        {
            "cluster_name": cluster_name,
            "etcd_encryption_type": etcd_type,
            "etcd_encrypted": etcd_ok,
            "default_sc_encrypted": default_sc_encrypted,
            "encrypted_storage_classes": encrypted_sc_count,
            "luks_machineconfigs": luks_count,
            "pvs_encrypted": f"{pv_encrypted}/{pv_total}",
            "ocp46_compliant": compliant,
        }
    )
df_ocp46_flags = pd.DataFrame(rows)
style_table(df_ocp46_flags, caption="OCP-46 — Compliance flags by cluster")

---
## OCP-47: Build / Source-to-Image (S2I) Policy

*Build pipelines on the cluster (BuildConfigs and S2I) follow a hardened policy.*

Evidence is sourced from the `build_s2i_policy` table:

- A cluster-scoped `build.config.openshift.io/cluster` is present and supplies
  build defaults (`buildDefaults.imageLabels`, resource limits/requests).
- No `BuildConfig` uses the `Custom` strategy with the docker socket exposed
  (`spec.strategy.customStrategy.exposeDockerSocket=true`), and no `BuildConfig`
  pulls every build (`forcePull=true` is restricted to controlled images).
- Every `ImageStream` carries `spec.lookupPolicy.local=true` so cluster
  workloads are resolved against the internal registry rather than arbitrary
  external mirrors.


In [ ]:
df_ocp47 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        BuildS2iPolicy.record_type,
        BuildS2iPolicy.name,
        BuildS2iPolicy.namespace,
        BuildS2iPolicy.detail_1,
        BuildS2iPolicy.detail_2,
        BuildS2iPolicy.detail_3,
        BuildS2iPolicy.detail_4,
        BuildS2iPolicy.detail_5,
        BuildS2iPolicy.detail_6,
    )
    .join(Cluster, BuildS2iPolicy.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        BuildS2iPolicy.record_type,
        BuildS2iPolicy.namespace,
        BuildS2iPolicy.name,
    )
    .statement,
    engine,
)
style_table(df_ocp47, caption="OCP-47 — Build / S2I Policy (raw records)")

### OCP-47: Compliance flags

A cluster is considered **compliant** when **all** of the following hold:

- A `build_default_config` row exists with non-empty image labels (`detail_2`).
- No `build_config` row uses the `Custom` strategy with the docker socket
  exposed (last field of `detail_6`, encoded as `<exposeDockerSocket>|<noCache>`,
  is `true`).
- Every `image_stream_policy` row has `lookupPolicy.local=true` (`detail_1`).


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp47[df_ocp47["cluster_name"] == cluster_name]
    bd = sub[sub["record_type"] == "build_default_config"]
    has_build_defaults = bool(
        ((bd["name"] != "(none)") & (bd["detail_2"].fillna("") != "")).any()
    )
    bcs = sub[sub["record_type"] == "build_config"]

    def _privileged(detail_6: object) -> bool:
        s = str(detail_6 or "").split("|")
        return len(s) >= 1 and s[0].strip().lower() == "true"

    privileged_bcs = (
        int(bcs["detail_6"].apply(_privileged).sum()) if not bcs.empty else 0
    )
    iss = sub[sub["record_type"] == "image_stream_policy"]
    if iss.empty:
        all_lookup_local = True
    else:
        all_lookup_local = bool(
            (iss["detail_1"].fillna("").str.lower() == "true").all()
        )

    compliant = has_build_defaults and privileged_bcs == 0 and all_lookup_local
    rows.append(
        {
            "cluster_name": cluster_name,
            "has_build_defaults": has_build_defaults,
            "privileged_buildconfigs": privileged_bcs,
            "all_imagestreams_lookup_local": all_lookup_local,
            "ocp47_compliant": compliant,
        }
    )

df_ocp47_flags = pd.DataFrame(rows)
style_table(df_ocp47_flags, caption="OCP-47 — Build / S2I Policy compliance flags")

---
## OCP-48: Image Signing & Verification

*Container images are signed and signatures are verified before deployment.*

Evidence is sourced from the `image-signing-verification` export. `record_type`
is one of:

- `cluster_image_policy_signature` — `ClusterImagePolicy` / `ImagePolicy` rules
  requiring `PublicKey` or `FulcioCAWithRekor` signatures, with their `scopes`
- `build_config` — OpenShift `BuildConfig` push targets and any cosign /
  notation signing hook
- `tekton_signing_task` — Tekton tasks / cluster-tasks named for cosign /
  notation / sigstore that perform signing as part of CI
- `registry_signature_config` — registry sources / sigstore signature stores
  declared on `image.config.openshift.io/cluster`


In [ ]:
df_ocp48 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ImageSigningVerification.record_type,
        ImageSigningVerification.name,
        ImageSigningVerification.namespace,
        ImageSigningVerification.detail_1,
        ImageSigningVerification.detail_2,
        ImageSigningVerification.detail_3,
        ImageSigningVerification.detail_4,
        ImageSigningVerification.detail_5,
        ImageSigningVerification.detail_6,
    )
    .join(Cluster, ImageSigningVerification.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        ImageSigningVerification.record_type,
        ImageSigningVerification.name,
    )
    .statement,
    engine,
)
style_table(df_ocp48, caption="OCP-48 — Image Signing & Verification (raw records)")

### OCP-48: Compliance flags

A cluster is considered **compliant** when **all** of the following hold:

- At least one `cluster_image_policy_signature` row has `detail_2=true`
  (`signature_required`) — signatures are verified at admission.
- At least one `tekton_signing_task` *or* a `build_config` with
  `signing_enabled=true` (`detail_4`) exists — pipelines actually sign images.
- At least one `registry_signature_config` row is present — the cluster knows
  where to fetch signatures from.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp48[df_ocp48["cluster_name"] == cluster_name]

    cip = sub[sub["record_type"] == "cluster_image_policy_signature"]
    sig_required_count = int((cip["detail_2"].fillna("").str.lower() == "true").sum())

    bc = sub[sub["record_type"] == "build_config"]
    signing_builds = int((bc["detail_4"].fillna("").str.lower() == "true").sum())

    tk = sub[sub["record_type"] == "tekton_signing_task"]
    signing_tasks = len(tk)

    reg = sub[sub["record_type"] == "registry_signature_config"]
    sig_registries = len(reg)

    compliant = (
        sig_required_count >= 1
        and (signing_builds >= 1 or signing_tasks >= 1)
        and sig_registries >= 1
    )
    rows.append(
        {
            "cluster_name": cluster_name,
            "policies_requiring_signature": sig_required_count,
            "signing_build_configs": signing_builds,
            "tekton_signing_tasks": signing_tasks,
            "registry_signature_stores": sig_registries,
            "ocp48_compliant": compliant,
        }
    )
df_ocp48_flags = pd.DataFrame(rows)
style_table(df_ocp48_flags, caption="OCP-48 — Compliance flags by cluster")

---
## OCP-49: Ephemeral Storage Limits

*Ephemeral storage is bounded so that pods cannot exhaust node disks.*

Evidence is sourced from the `ephemeral_storage_limits` table — one row per
namespace capturing whether a `LimitRange` supplies a default request / default
limit / max for `ephemeral-storage`, whether a `ResourceQuota` caps
`requests.ephemeral-storage` and `limits.ephemeral-storage`, and how many pods
in the namespace use `emptyDir` volumes without `sizeLimit`.


In [ ]:
df_ocp49 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        EphemeralStorageLimits.namespace,
        EphemeralStorageLimits.is_system_namespace,
        EphemeralStorageLimits.has_lr_default_request,
        EphemeralStorageLimits.has_lr_default_limit,
        EphemeralStorageLimits.has_lr_max,
        EphemeralStorageLimits.lr_default_request,
        EphemeralStorageLimits.lr_default_limit,
        EphemeralStorageLimits.lr_max,
        EphemeralStorageLimits.has_quota_request,
        EphemeralStorageLimits.has_quota_limit,
        EphemeralStorageLimits.quota_request_hard,
        EphemeralStorageLimits.quota_limit_hard,
        EphemeralStorageLimits.emptydir_pods_total,
        EphemeralStorageLimits.emptydir_pods_without_sizelimit,
    )
    .join(Cluster, EphemeralStorageLimits.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, EphemeralStorageLimits.namespace)
    .statement,
    engine,
)
style_table(df_ocp49, caption="OCP-49 — Ephemeral Storage Limits (raw records)")

### OCP-49: Compliance flags

A cluster is considered **compliant** when, for every **user** namespace
(non-`openshift-*`, non-`kube-*`, non-`default`), **all** of the following
hold:

- `LimitRange` supplies both a default request and a default limit for
  `ephemeral-storage` (`has_lr_default_request=true`,
  `has_lr_default_limit=true`).
- A `ResourceQuota` caps `requests.ephemeral-storage`
  (`has_quota_request=true`).
- No pod uses an `emptyDir` volume without `sizeLimit`
  (`emptydir_pods_without_sizelimit=0`).


In [ ]:
def _b49(s: object) -> bool:
    return str(s or "").strip().lower() == "true"


rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp49[df_ocp49["cluster_name"] == cluster_name]
    user = sub[sub["is_system_namespace"].fillna("").str.lower() != "true"]
    if user.empty:
        ns_total = 0
        ns_with_defaults = 0
        ns_with_quota = 0
        ns_with_unbounded = 0
    else:
        ns_total = len(user)
        ns_with_defaults = int(
            (
                user["has_lr_default_request"].apply(_b49)
                & user["has_lr_default_limit"].apply(_b49)
            ).sum()
        )
        ns_with_quota = int(user["has_quota_request"].apply(_b49).sum())
        ns_with_unbounded = int(
            (
                pd.to_numeric(
                    user["emptydir_pods_without_sizelimit"], errors="coerce"
                ).fillna(0)
                > 0
            ).sum()
        )

    compliant = (
        ns_total > 0
        and ns_with_defaults == ns_total
        and ns_with_quota == ns_total
        and ns_with_unbounded == 0
    )
    rows.append(
        {
            "cluster_name": cluster_name,
            "user_namespaces": ns_total,
            "with_lr_defaults": ns_with_defaults,
            "with_quota_request": ns_with_quota,
            "with_unbounded_emptydir": ns_with_unbounded,
            "ocp49_compliant": compliant,
        }
    )

df_ocp49_flags = pd.DataFrame(rows)
style_table(
    df_ocp49_flags, caption="OCP-49 — Ephemeral Storage Limits compliance flags"
)

---
## OCP-50: Admission Controller Hardening

*Admission controllers (built-in and webhook) are configured to fail safe.*

Evidence is sourced from the `admission_controller_hardening` table:

- The OCP 4.18 default-on hardening plugins — `LimitRanger`, `ResourceQuota`,
  `PodSecurity`, `NodeRestriction`, `MutatingAdmissionWebhook`,
  `ValidatingAdmissionWebhook` — are not explicitly disabled.
- No `ValidatingWebhookConfiguration` or `MutatingWebhookConfiguration` is
  registered with `failurePolicy=Ignore` **and** `sideEffects=Unknown` (a
  combination that allows requests through silently when the webhook is
  unreachable and gives no signal that mutations occurred).


In [ ]:
df_ocp50 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        AdmissionControllerHardening.record_type,
        AdmissionControllerHardening.name,
        AdmissionControllerHardening.namespace,
        AdmissionControllerHardening.detail_1,
        AdmissionControllerHardening.detail_2,
        AdmissionControllerHardening.detail_3,
        AdmissionControllerHardening.detail_4,
        AdmissionControllerHardening.detail_5,
        AdmissionControllerHardening.detail_6,
    )
    .join(Cluster, AdmissionControllerHardening.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        AdmissionControllerHardening.record_type,
        AdmissionControllerHardening.name,
    )
    .statement,
    engine,
)
style_table(df_ocp50, caption="OCP-50 — Admission Controller Hardening (raw records)")

### OCP-50: Compliance flags

A cluster is considered **compliant** when **all** of the following hold:

- For each of `PodSecurity`, `LimitRanger`, `ResourceQuota`, `NodeRestriction`,
  `MutatingAdmissionWebhook`, `ValidatingAdmissionWebhook` the
  `default_admission_plugin` row reports `enabled` or `default` (i.e. not
  `disabled`).
- No `validating_webhook` or `mutating_webhook` row combines
  `failurePolicy=Ignore` with `sideEffects=Unknown`.


In [ ]:
REQUIRED_PLUGINS = {
    "PodSecurity",
    "LimitRanger",
    "ResourceQuota",
    "NodeRestriction",
    "MutatingAdmissionWebhook",
    "ValidatingAdmissionWebhook",
}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp50[df_ocp50["cluster_name"] == cluster_name]
    plugins = sub[sub["record_type"] == "default_admission_plugin"]
    disabled_required = set()
    for _, r in plugins.iterrows():
        if (
            r["name"] in REQUIRED_PLUGINS
            and str(r["detail_1"] or "").strip().lower() == "disabled"
        ):
            disabled_required.add(r["name"])
    no_disabled_required = len(disabled_required) == 0

    webhooks = sub[sub["record_type"].isin(["validating_webhook", "mutating_webhook"])]
    risky = webhooks[
        (webhooks["detail_1"].fillna("").str.lower() == "ignore")
        & (webhooks["detail_3"].fillna("").str.lower() == "unknown")
    ]
    risky_webhooks = int(len(risky))

    compliant = no_disabled_required and risky_webhooks == 0
    rows.append(
        {
            "cluster_name": cluster_name,
            "disabled_required_plugins": ";".join(sorted(disabled_required)),
            "risky_webhooks": risky_webhooks,
            "ocp50_compliant": compliant,
        }
    )

df_ocp50_flags = pd.DataFrame(rows)
style_table(
    df_ocp50_flags, caption="OCP-50 — Admission Controller Hardening compliance flags"
)

In [ ]:
session.close()
print("Session closed.")